# 02. 사출성형 데이터 EDA와 OOF 오류 분석

`Python (kamp-base)` 커널을 선택하고 `Shift+Enter`로 위에서 아래로 실행하세요.
기본은 **생성된 EDA 결과 확인**입니다. 새 진단을 실행하려면 `RUN_EDA = True`로 바꿉니다.

이번 단계는 **데이터 이해와 오류 분석**입니다. 모델 재학습, 원본 수정, 파생변수 추가는 하지 않습니다.
`class_1`이 불량이라는 의미는 미확인입니다. 아래에서 ‘양성’은 숫자 1을 뜻합니다.

전체 라벨을 보고 탐색한 변수/구간 관계는 독립적인 평가 성능도, 공정의 인과관계도 아닙니다.

## 1. 환경과 경로

In [ ]:
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "1.injection-molding" / "configs" / "baseline_v2.json").exists()), None)
if ROOT is None:
    raise RuntimeError("kamp-ai 프로젝트 안에서 Jupyter를 실행하세요.")
PROJECT = ROOT / "1.injection-molding"
print("Python:", sys.executable)
if Path(sys.prefix).name != "kamp-base":
    raise RuntimeError("Python (kamp-base) 커널을 선택하세요.")
BASELINE = PROJECT / "artifacts" / "baseline_v2"
EXISTING_EDA = PROJECT / "artifacts" / "eda_v1"


## 2. 저장된 결과 보기 / EDA 재실행

- `RUN_EDA = False`: 이미 생성된 표와 그림을 읽습니다.
- `RUN_EDA = True`: baseline_v2의 저장된 OOF 예측과 원본 CSV로 새 EDA를 생성합니다.
- `DOWNLOAD_OFFICIAL_SOURCE = True`: 공식 ZIP·과제문서를 다시 내려받아 로컬 CSV와 비교합니다. 원본 CSV를 덮어쓰거나 압축을 풀지 않습니다.

GitHub에서 처음 복제했다면 원본 데이터와 baseline_v2의 로컬 OOF 파일을 먼저 준비해야 새 EDA를 실행할 수 있습니다. 원본·모델·행별 예측은 공개 저장소에 포함하지 않습니다.

In [ ]:
RUN_EDA = False
DOWNLOAD_OFFICIAL_SOURCE = False
SOURCE_CHECK = EXISTING_EDA / "source_check.json"

def run_command(args):
    result = subprocess.run([sys.executable, "-u", *map(str, args)], cwd=ROOT,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    result.check_returncode()

if DOWNLOAD_OFFICIAL_SOURCE:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
    SOURCE_CHECK = PROJECT / "artifacts" / "source_checks" / (timestamp + ".json")
    run_command([PROJECT / "scripts" / "verify_source.py", "--download", "--output", SOURCE_CHECK])

if RUN_EDA:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
    EDA_DIR = PROJECT / "artifacts" / "eda_runs" / timestamp
    command = [PROJECT / "scripts" / "eda_analysis.py", "--baseline-dir", BASELINE,
               "--output-dir", EDA_DIR]
    if SOURCE_CHECK.exists():
        command += ["--source-check", SOURCE_CHECK]
    run_command(command)
else:
    EDA_DIR = EXISTING_EDA

manifest = json.loads((EDA_DIR / "manifest.json").read_text())
print("현재 결과:", EDA_DIR)
display(manifest)

def table(name):
    return pd.read_csv(EDA_DIR / name)

def figure(name, width=1100):
    display(Image(filename=str(EDA_DIR / "figures" / name), width=width))


## 3. 공식 ZIP과 로컬 데이터가 같은가?

SHA-256과 파일 바이트를 비교합니다. 일치하면 로컬 편집 때문에 중복·충돌이 생겼다는 가능성은 배제할 수 있습니다.
**일치한다고 해서 라벨 자체의 정확성이나 생성 과정을 검증한 것은 아닙니다.**

In [ ]:
source_file = EDA_DIR / "source_check.json"
if source_file.exists():
    source = json.loads(source_file.read_text())
    display(pd.DataFrame(source["files"])[["file", "bytes", "bytes_match"]])
    print("라벨 의미:", source["label_semantics"])
    print("과제공개 문서 키워드 검색:", source.get("task_document_term_present"))
else:
    print("이번 실행에는 공식 파일 비교 기록이 없습니다.")


## 4. 클래스 비율, 중복, 행 순서

**주의할 발견:** CN7의 class_1이 파일 앞 약 10%에 모여 있습니다. 행 순서를 생산시간이라고 가정하지 않습니다.
‘앞부분의 공정 특성’을 학습한 효과와 ‘품질을 구분한 효과’가 섞였을 가능성을 살펴봅니다. ID/행 번호는 모델 입력으로 사용하지 않습니다.

그림 위쪽은 파일을 순서대로 10구간으로 나눈 class_1 개수입니다. 아래쪽의 빨간 점은 class_1입니다.

In [ ]:
display(table("data_summary.csv"))
display(table("row_blocks.csv"))
figure("01_row_order.png")


## 5. 어떤 변수가 클래스를 구분하는가?

`auc_direction_free`는 **변수 하나와 전체 라벨의 탐색적 관계**입니다. 모델의 교차검증 AP와는 다른 값입니다.
0.5에 가까우면 해당 변수 하나로 순위를 구분하기 어렵고, 1에 가까우면 잘 구분되는 모습입니다.
전체 라벨을 보고 변수/방향을 골랐으므로 높게 나와도 검증 성능이라고 해석하지 않습니다.

`first_decile_auc_direction_free`는 앞 10% 안에서만 비교한 값입니다. RG3는 이 구간에 class_1이 없어 NaN(계산 불가)입니다.

ECDF 그래프의 세로축은 **각 클래스 내부의 누적 비율**입니다. 선이 겹칠수록 분포가 비슷합니다.

In [ ]:
feature_stats = table("feature_class_summary.csv")
for product in ["cn7", "rg3"]:
    print(product.upper())
    display(feature_stats[(feature_stats.product_group == product) & ~feature_stats.constant]
            .nlargest(8, "auc_direction_free")
            [["feature", "class_0_mean", "class_1_mean", "auc_direction_free",
              "first_decile_auc_direction_free"]].round(3))
figure("02_class_distributions.png")


## 6. 변수끼리 비슷한 정보를 담고 있는가?

상관계수가 1 또는 -1에 가까우면 두 변수가 함께 움직이는 경향이 강합니다.
이는 인과관계가 아니며, 트리 모델에서 상관 높다는 이유만으로 반드시 삭제해야 하는 것도 아닙니다.

In [ ]:
correlations = table("correlated_pairs.csv")
display(correlations.assign(abs_correlation=correlations.correlation.abs())
        .sort_values("abs_correlation", ascending=False).head(20))
figure("03_correlations.png", width=1300)


## 7. labeled와 unlabeled의 분포는 같은가?

KS 거리는 두 누적분포의 최대 차이로, 0이면 같고 클수록 차이가 큽니다. 여기서는 p-value를 사용하지 않습니다.
평균 0·표준편차 1이어도 분포 모양은 다를 수 있습니다.

파일별 표준화 정황이 있어 동일 수치가 같은 원 단위를 뜻하지 않습니다. 범위 밖 비율은 **제공된 값의 범위**에 대한 기술 통계입니다.
`Clamp_Open_Position`은 labeled에서 상수이므로 이미 baseline에서 제외했습니다.

In [ ]:
drift = table("distribution_shift.csv")
for product in ["cn7", "rg3"]:
    display(drift[drift.product_group == product].nlargest(8, "ks_statistic")
            [["product_group", "feature", "ks_statistic", "wasserstein_standardized",
              "unlabeled_outside_labeled_range"]].round(3))
figure("04_distribution_shift.png")
display(table("outlier_summary.csv"))


## 8. 오류를 어떻게 셌는가?

저장된 **OOF 예측**만 사용합니다. 각 행은 자신을 학습하지 않은 모델의 예측입니다.

- `inner_f1`: 학습 데이터 안에서 선택한 임계값으로 0·1 판정
- `top10`: 각 외부 평가 fold에서 점수 상위 10%만 검사
- FN: 실제 class_1인데 판정이 0 (또는 해당 검사 정책에서 미선택)
- FP: 실제 class_0인데 판정이 1 (또는 해당 검사 정책에서 선택)

세 반복을 원본 행이 세 배 늘어난 것으로 집계하지 않습니다. 아래 표는 한 seed만 보여줍니다.
충돌 없는 RG3 구간에는 class_1이 없으므로 Recall은 NaN입니다.

In [ ]:
PRODUCT = "cn7"  # "rg3"
MODEL = "catboost_balanced"  # logistic, xgboost_balanced, extratrees, lightgbm_balanced 등
SEED = 42

cohorts = table("error_cohorts_by_seed.csv")
print("사용 가능한 모델:", cohorts.model.unique().tolist())
display(cohorts[(cohorts.scenario == PRODUCT) & (cohorts.product_group == PRODUCT)
                & (cohorts.model == MODEL) & (cohorts.seed == SEED)]
        [["cohort", "policy", "rows", "positives", "tp", "fp", "fn", "tn", "precision", "recall"]])


## 9. 같은 행을 반복적으로 놓치는가?

`missed_all_seeds`: 동일한 원본 class_1을 세 seed 모두 0으로 판정한 행 수입니다.
`outside_top10_all_seeds`: 세 seed 모두 검사 상위 10%에 포함되지 않은 행 수입니다.
둘은 다른 문제입니다. **판정 임계값에서 놓치더라도 검사 우선순위에서는 찾을 수 있습니다.**

In [ ]:
stable = table("stable_misses.csv")
display(stable[stable.product_group == PRODUCT])
consensus_path = EDA_DIR / "rows" / "class1_consensus.csv"
if consensus_path.exists():
    consensus = pd.read_csv(consensus_path)
    display(consensus[consensus.product_group == PRODUCT]
            .sort_values("models_miss_all_seeds", ascending=False))
else:
    print("행별 결과는 Git에서 제외되어 있습니다. RUN_EDA=True로 로컬 재생성하세요.")


## 10. 모델별로 놓치는 행을 그림으로 보기

아래는 **실제 class_1만** 모은 seed 42의 예측입니다.
초록색은 발견, 빨간색은 미탐입니다. 왼쪽은 임계값 판정, 오른쪽은 상위 10% 검사입니다.
행 이름은 원본의 0부터 시작하는 행 위치이며, `*`는 동일 공정값에 다른 라벨이 있는 그룹입니다.

In [ ]:
figure("05_class1_error_matrix.png", width=1250)


## 11. 원본 공정값과 실제 오류 행 함께 보기

대표 5개 모델의 seed 42 OOF를 원본 공정 특성과 결합했습니다. `PassOrFail`, `label_conflict`, 모델 점수, 임계값, 검사 선택 여부를 함께 보세요.
행을 고르고 싶으면 `ROW_POSITION`에 앞의 그림에서 본 번호를 입력합니다. 이 셀은 진단 표를 읽을 뿐 원본을 바꾸지 않습니다.

In [ ]:
ROW_POSITION = None  # 예: 47 (0부터 시작하는 행 번호)
case_path = EDA_DIR / "rows" / "focus_oof_with_features.csv"
if case_path.exists():
    cases = pd.read_csv(case_path)
    part = cases[(cases.product_group == PRODUCT) & (cases.model == MODEL)]
    if ROW_POSITION is None:
        part = part[(part.PassOrFail == 1) & (part.prediction_inner_f1 == 0)]
    else:
        part = part[part.row_position == ROW_POSITION]
    display(part.sort_values("p_class_1").head(15))
else:
    print("행별 결과가 없습니다. RUN_EDA=True로 생성하세요.")


## 12. 어떤 변수 구간에서 오류가 많은가?

구간은 표준화 값 기준으로 미리 고정했습니다. 아래 표는 전체 라벨에서 찾아낸 ‘확정 규칙’이 아닙니다.
표본 수(`rows`)와 class_1 수(`positives`)를 반드시 함께 보세요. 양성이 없으면 Recall을 계산할 수 없습니다.

In [ ]:
FEATURE = "Mold_Temperature_3"
bins = table("error_feature_bins_seed42.csv")
display(bins[(bins.product_group == PRODUCT) & (bins.model == MODEL) & (bins.feature == FEATURE)])
block_errors = table("errors_by_row_block_seed42.csv")
display(block_errors[(block_errors.product_group == PRODUCT) & (block_errors.model == MODEL)])


## 13. 종합 보고서와 다음 단계

확인한 사실과 가설을 구분합니다. CN7의 파일 구간 집중, RG3의 약한 구분, 충돌 라벨, 미라벨 분포 차이를 먼저 이해한 뒤 소수의 파생변수를 비교합니다.
행 번호를 입력으로 쓰거나, class_1 극단값을 자동 삭제하거나, 충돌 라벨을 임의로 수정하지 않습니다.

In [ ]:
display(Markdown((EDA_DIR / "report.md").read_text()))
